[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OlivierGeorgeon/bac_a_sable/blob/master/naive_bayes.ipynb)

# Algorithme Bayésien Naïf

# Theorem de Bayes

La probabilité qu'un évenement A survienne quand un événement B est survenu est égal à la probabilité que B survienne si A survient multiplié par la probabilité de B, le tout divisé par la probabilité de A.

$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$

Par exemple la probabilité qu'il fasse beau chez votre grand mère (événement A) quand elle vous appelle (événement B) augmente avec la probabilité qu'elle vous appelle quand il fait beau, et avec la probabilité qu'il faisse beau là ou elle vit, et diminue avec la probabilité qu'elle vous appelle qu'il fasse beau ou non. 

## Application au machine learning

On note le *feature vector* $X = (x_1, x_2, ..., x_n)$. 
On calcule la probabilité d'une mesure $y$:

$P(y|X) = \frac{P(X|y) \cdot P(y)}{P(X)}$

$P(y|X)$ est la probability postérieure

$P(X|y)$ est la probabilité conditionnelle

$P(y)$ et $P(X)$ sont les prior probabilité

L'approche naive suppose que touts les *features* sont indépendants. 
Par exemple $x_1$ est un appel de votre grand mère et $x_2$ est un appel de sa voisine. Les *features* sont indépendants si elles ne se concertent pas pour vous appeler. 
Dans ce cas on peut éclater les probabilités:

$P(y|X) = \frac{P(x_1|y) \cdot P(x_2|y) \cdot ... \cdot P(x_n|y) \cdot P(y)}{P(X)}$


# Faire un mécanisme de classification en utilisant la formule de Bayes

Nous voulons déterminer la catégorie la plus probable d'un objet qui est décrit par le vecteur de features $X = (x_1, x_2, ..., x_n)$.

Par exemple les catégorie "il fait beau" et "il fait maussade" ou "il fait mauvais temps" chez votre grand mère. 

Il faut calculer la probabilité pour chacune des trois catégories et trouver le maximum. 

On note celci en utilisant la fonction $argmax_x(f(x))$ qui renvoie la valeur de x pour laquelle $f(x)$ est maximum:

$y = argmax_yP(y|X) = argmax_y \frac{P(x_1|y) \cdot P(x_2|y) \cdot ... \cdot P(x_n|y) \cdot P(y)}{P(X)}$ 

Pour simplifier les calculs, on prend le log, ce qui revient au même car la fonction log est croissante. Nous pouvons aussi supprimer $P(X)$ car c'est une constante:

$y = argmax_y \log(P(x_1|y)) + \log(P(x_2|y)) + ... + \log(P(x_n|y) + \log(P(y))$

## Nous calculons la "prior probability" $P(y)$.

 C'est la fréquence de chaque catégorie indépendement des 
 
 
 de temps indépendement des appels de votre grand mère.

# On code

In [6]:
import numpy as np

La méthode `fit(X, y)` calcule les probabilités $P(x_i|y)$ de chaque catégorie en fonction de chaque features. 

On suppose que cette probabilité est décrite par une fonction gaussienne qu'on caractérise par sa moyenne et son écart type.

Par exemple, il fait beau 8 fois sur 10 quand votre grand mère appelle, avec un écart type de 1, et seulement 7 fois sur 10 quand la voisine appelle.

La méthode `_pdf()` calcule la distribution de probabilité donnée par le gaussienne 

Il nous faut la distribution de probabilité d'une gaussiènne: 

$P(x_i|y) = \frac{1}{\sqrt{2 \pi \sigma_y^2}} \cdot exp(- \frac{(x_i - \mu_y)^2}{2 \sigma_y^2}) \$

In [5]:
class NaiveBayes:
    def fit(self, X, y):
        """"""
        # X un tableau a deux dimensions: les features en colonnes et les échantillons en ligne
        n_samples, n_features = X.shape
        # y est la catégorie de chaque échantillon
        # On récupère la liste des catégories (classes) existantes dans nos échantillons
        self._classes = np.unique(y) 
        n_classes = len(self._classes)

        # calculate mean, var, and prior for each class
        self._mean = np.zeros((n_classes, n_features), dtype=np.float64)
        self._var = np.zeros((n_classes, n_features), dtype=np.float64)
        self._priors = np.zeros(n_classes, dtype=np.float64)

        for idx, c in enumerate(self._classes):
            X_c = X[y == c]
            self._mean[idx, :] = X_c.mean(axis=0)
            self._var[idx, :] = X_c.var(axis=0)
            self._priors[idx] = X_c.shape[0] / float(n_samples)

    def predict(self, X):
        """Predit pour tous les échanitllons"""
        y_pred = [self._predict(x) for x in X]
        return np.array(y_pred)

    def _predict(self, x):
        """Predit pour un échantillon"""
        posteriors = []

        # calculate posterior probability for each class
        for idx, c in enumerate(self._classes):
            prior = np.log(self._priors[idx])
            posterior = np.sum(np.log(self._pdf(idx, x)))
            posterior = prior + posterior
            posteriors.append(posterior)

        # return class with highest posterior probability
        return self._classes[np.argmax(posteriors)]

    def _pdf(self, class_idx, x):
        """Probability density function"""
        mean = self._mean[class_idx]
        var = self._var[class_idx]
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator


# On lance l'entrainement

In [7]:
# Imports
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import datasets

def accuracy(y_true, y_pred):
    accuracy = np.sum(y_true == y_pred) / len(y_true)
    return accuracy

X, y = datasets.make_classification(
    n_samples=1000, n_features=10, n_classes=2, random_state=123
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

nb = NaiveBayes()
nb.fit(X_train, y_train)
predictions = nb.predict(X_test)

print("Naive Bayes classification accuracy", accuracy(y_test, predictions))

Naive Bayes classification accuracy 0.965


# Activité

Créer votre propre jeu de données d'entrainement et refaites le test avec des nouveaux échantillons